# Exploratory Data Analysis: Dataset Statistics

This notebook performs comprehensive exploratory data analysis (EDA) on the hierarchical multimodal multilabel video dataset.

### Key Objectives:
1. **Label Analysis**: Examine class imbalance, prevalence of normal content versus 7 harm categories, multi-label cardinality, and label co-occurrences.
2. **Video Technical Analysis**: Extract and analyze video technical properties (duration, resolution, frame rate, aspect ratio, file size) across video partition subdirectories (`part_01` to `part_13`).
3. **Integrity & Outlier Detection**: Identify unreadable/corrupted files and flag long duration outliers (>180s/300s).
4. **Figure Persistence**: Export high-resolution visualization figures to `images/eda/` for documentation and reporting.


In [ ]:
from pathlib import Path
import os
import time
from concurrent.futures import ThreadPoolExecutor
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

# Configure plot styling
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]

# Define repository paths relative to this notebook
notebook_dir = Path.cwd()
# Handle execution whether run from src/eda or project root
repo_root = notebook_dir if (notebook_dir / "data").exists() else notebook_dir.parent.parent

csv_path = repo_root / "data" / "label-origin.csv"
video_parts_dir = repo_root / "data" / "video_parts"
metadata_cache_path = repo_root / "data" / "video_metadata.csv"
figures_dir = repo_root / "images" / "eda"
figures_dir.mkdir(parents=True, exist_ok=True)

print(f"Repo root       : {repo_root}")
print(f"Labels CSV      : {csv_path} (exists: {csv_path.exists()})")
print(f"Video parts dir : {video_parts_dir} (exists: {video_parts_dir.exists()})")
print(f"Figures dir     : {figures_dir}")


## 1. Dataset Loading and Ground-Truth Integrity Check

We load `data/label-origin.csv` and verify alignment with all `.mp4` video files present across `data/video_parts/part_*`.


In [ ]:
# Load ground-truth labels
df_labels = pd.read_csv(csv_path)
print(f"Loaded ground-truth CSV: {len(df_labels)} samples, {df_labels.shape[1]} columns")

# Identify canonical harm categories
harm_cols = [c for c in df_labels.columns if c != "filename"]

# Ensure binary integer types
for col in harm_cols:
    df_labels[col] = pd.to_numeric(df_labels[col], errors="coerce").fillna(0).astype(int)

# Derive normal_content indicator (1 if zero harms present, else 0)
df_labels["normal_content"] = (df_labels[harm_cols].sum(axis=1) == 0).astype(int)
df_labels["is_harmful"] = (1 - df_labels["normal_content"]).astype(int)
df_labels["total_harms"] = df_labels[harm_cols].sum(axis=1)

# Scan video parts directory
existing_video_paths = {p.name: p for p in video_parts_dir.rglob("*.mp4")}
print(f"Total video files found in parts: {len(existing_video_paths)}")

# Verify 1:1 correspondence
missing_videos = set(df_labels["filename"]) - set(existing_video_paths.keys())
unlabeled_videos = set(existing_video_paths.keys()) - set(df_labels["filename"])

print(f"Missing video files in storage: {len(missing_videos)}")
print(f"Unlabeled video files in parts : {len(unlabeled_videos)}")
if len(missing_videos) == 0 and len(unlabeled_videos) == 0:
    print("SUCCESS: Perfect 1:1 match between CSV entries and video files on disk.")


## 2. Label Distribution and Class Prevalence

We evaluate the distribution of `normal_content` against each of the 7 harm categories:
- `information_harm`
- `sexual_harm`
- `psychological_harm`
- `hate_harassment_harm`
- `clickbait_harm`
- `addictive_harm`
- `physical_harm`


In [ ]:
n_total = len(df_labels)
all_eval_cols = ["normal_content"] + harm_cols
counts = df_labels[all_eval_cols].sum()
prevalences = (counts / n_total) * 100

stats_df = pd.DataFrame({
    "Category": all_eval_cols,
    "Count": counts.values,
    "Prevalence (%)": prevalences.values.round(2)
}).sort_values(by="Count", ascending=False).reset_index(drop=True)

display(stats_df)

# Plot label distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = ["#2b5c8f" if cat == "normal_content" else "#c94c4c" for cat in stats_df["Category"]]
sns.barplot(data=stats_df, x="Count", y="Category", palette=colors, ax=axes[0])
axes[0].set_title("Sample Count per Category", fontweight="bold")
axes[0].set_xlabel("Number of Samples")
for i, count in enumerate(stats_df["Count"]):
    axes[0].text(count + 15, i, f"{count:,}", va="center", fontsize=10)

sns.barplot(data=stats_df, x="Prevalence (%)", y="Category", palette=colors, ax=axes[1])
axes[1].set_title("Prevalence Percentage (%)", fontweight="bold")
axes[1].set_xlabel("Percentage of Total Dataset (%)")
for i, prev in enumerate(stats_df["Prevalence (%)"]):
    axes[1].text(prev + 0.5, i, f"{prev:.2f}%", va="center", fontsize=10)

plt.tight_layout()
fig_path = figures_dir / "label_distribution.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved figure: {fig_path}")


## 3. Multi-label Cardinality and Label Combinations

We inspect the distribution of harm labels assigned per sample to quantify single-label vs multi-label complexity.


In [ ]:
cardinality_counts = df_labels["total_harms"].value_counts().sort_index()
cardinality_pct = (cardinality_counts / n_total * 100).round(2)

cardinality_df = pd.DataFrame({
    "Num Harms": cardinality_counts.index,
    "Count": cardinality_counts.values,
    "Percentage (%)": cardinality_pct.values
})
display(cardinality_df)

# Visualize label cardinality
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=cardinality_df, x="Num Harms", y="Count", palette="Blues_d", ax=ax)
ax.set_title("Distribution of Number of Harm Labels per Video", fontweight="bold")
ax.set_xlabel("Number of Concurrent Harm Labels")
ax.set_ylabel("Number of Videos")
for p in ax.patches:
    val = int(p.get_height())
    pct = (val / n_total) * 100
    ax.annotate(f"{val:,}\n({pct:.1f}%)", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha="center", va="center", color="white" if val > 500 else "black", fontweight="bold")

plt.tight_layout()
fig_path = figures_dir / "label_cardinality.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved figure: {fig_path}")

# Examine top multi-label combinations
def get_combination(row):
    active = [c for c in harm_cols if row[c] == 1]
    return " + ".join(active) if active else "normal_content"

df_labels["label_combination"] = df_labels.apply(get_combination, axis=1)
combo_counts = df_labels["label_combination"].value_counts().head(15)

print("Top 15 Label Combinations in Dataset:")
combo_df = pd.DataFrame({"Combination": combo_counts.index, "Count": combo_counts.values})
display(combo_df)


## 4. Label Co-occurrence Analysis

We compute the pairwise co-occurrence matrix among the 7 harm categories on videos with at least one harm.


In [ ]:
harm_sub = df_labels[harm_cols]
co_occurrence = harm_sub.T.dot(harm_sub)

# Display raw co-occurrence counts
display(co_occurrence)

# Plot co-occurrence heatmap
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(co_occurrence, annot=True, fmt="d", cmap="YlOrRd", cbar=True, ax=ax, linewidths=0.5)
ax.set_title("Pairwise Co-occurrence Matrix (7 Harm Categories)", fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
fig_path = figures_dir / "label_cooccurrence_matrix.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved figure: {fig_path}")


## 5. Video Technical Metadata Extraction and Caching

Extracts technical attributes from each video file:
- `duration_sec`: Duration in seconds
- `width`, `height`: Pixel dimensions
- `aspect_ratio`: Classified ratio (9:16 vertical, 16:9 horizontal, 1:1, or custom)
- `fps`: Frame rate
- `frame_count`: Total number of frames
- `file_size_mb`: File size on disk in megabytes
- `is_valid`: Boolean flag checking whether OpenCV can read the file

Extraction is parallelized via `ThreadPoolExecutor` and cached to `data/video_metadata.csv`.


In [ ]:
def extract_video_metadata(filepath):
    """Extract technical properties of a video using OpenCV."""
    path = Path(filepath)
    filename = path.name
    try:
        size_mb = round(os.path.getsize(path) / (1024 * 1024), 2)
    except Exception:
        size_mb = 0.0

    try:
        cap = cv2.VideoCapture(str(path))
        if not cap.isOpened():
            return {
                "filename": filename,
                "video_path": str(path.relative_to(repo_root / "data")),
                "part": path.parent.name,
                "duration_sec": 0.0,
                "width": 0,
                "height": 0,
                "aspect_ratio": "unknown",
                "fps": 0.0,
                "frame_count": 0,
                "file_size_mb": size_mb,
                "is_valid": False,
                "error": "Failed to open video container"
            }

        fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
        frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0.0
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
        cap.release()

        duration = round(frame_count / fps, 2) if (fps > 0 and frame_count > 0) else 0.0

        ar_label = f"{width}:{height}"
        if width > 0 and height > 0:
            ratio = width / height
            if abs(ratio - 9/16) < 0.05:
                ar_label = "9:16 (Vertical)"
            elif abs(ratio - 16/9) < 0.05:
                ar_label = "16:9 (Horizontal)"
            elif abs(ratio - 1.0) < 0.05:
                ar_label = "1:1 (Square)"
            else:
                ar_label = f"{width}x{height} (Other)"

        return {
            "filename": filename,
            "video_path": str(path.relative_to(repo_root / "data")),
            "part": path.parent.name,
            "duration_sec": duration,
            "width": width,
            "height": height,
            "aspect_ratio": ar_label,
            "fps": round(fps, 2),
            "frame_count": int(frame_count),
            "file_size_mb": size_mb,
            "is_valid": True,
            "error": ""
        }
    except Exception as e:
        return {
            "filename": filename,
            "video_path": str(path.relative_to(repo_root / "data")),
            "part": path.parent.name,
            "duration_sec": 0.0,
            "width": 0,
            "height": 0,
            "aspect_ratio": "unknown",
            "fps": 0.0,
            "frame_count": 0,
            "file_size_mb": size_mb,
            "is_valid": False,
            "error": str(e)
        }

# Load cache if exists, otherwise extract
if metadata_cache_path.exists():
    print(f"Loading cached video metadata from: {metadata_cache_path}")
    df_meta = pd.read_csv(metadata_cache_path)
else:
    print("Extracting video metadata across all video parts...")
    all_video_files = sorted(list(video_parts_dir.rglob("*.mp4")))
    start_t = time.time()
    with ThreadPoolExecutor(max_workers=8) as executor:
        meta_records = list(executor.map(extract_video_metadata, all_video_files))
    df_meta = pd.DataFrame(meta_records)
    df_meta.to_csv(metadata_cache_path, index=False)
    print(f"Extraction completed in {time.time() - start_t:.2f}s and saved to {metadata_cache_path}")

print(f"Total metadata rows: {len(df_meta)}")
display(df_meta.head())


## 6. Video Duration Distribution and Summary

Analysis of video durations in seconds, including standard percentiles (25%, 50%, 75%, 90%, 95%, 99%).


In [ ]:
valid_meta = df_meta[df_meta["is_valid"] & (df_meta["duration_sec"] > 0)]
duration_series = valid_meta["duration_sec"]

desc = duration_series.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).to_frame("Duration (s)")
display(desc)

# Visualize duration distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

# Full histogram with KDE
sns.histplot(duration_series, bins=50, kde=True, color="#1d3557", ax=axes[0])
axes[0].set_title("Video Duration Distribution (All Videos)", fontweight="bold")
axes[0].set_xlabel("Duration (seconds)")
axes[0].set_ylabel("Video Count")
axes[0].axvline(duration_series.median(), color="red", linestyle="--", label=f"Median: {duration_series.median():.1f}s")
axes[0].axvline(duration_series.mean(), color="orange", linestyle=":", label=f"Mean: {duration_series.mean():.1f}s")
axes[0].legend()

# Zoom-in on typical short-form range (<= 120s)
sns.histplot(duration_series[duration_series <= 120], bins=40, kde=True, color="#457b9d", ax=axes[1])
axes[1].set_title("Video Duration Distribution (<= 120 seconds)", fontweight="bold")
axes[1].set_xlabel("Duration (seconds)")
axes[1].set_ylabel("Video Count")

plt.tight_layout()
fig_path = figures_dir / "video_duration_distribution.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved figure: {fig_path}")


## 7. Resolution and Aspect Ratio Distribution

TikTok and short-form video models are typically optimized for vertical 9:16 content. We inspect frame resolutions and orientations.


In [ ]:
ar_counts = valid_meta["aspect_ratio"].value_counts().reset_index()
ar_counts.columns = ["Aspect Ratio", "Count"]
ar_counts["Percentage (%)"] = (ar_counts["Count"] / len(valid_meta) * 100).round(2)
display(ar_counts)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Aspect ratio bar chart
sns.barplot(data=ar_counts, x="Count", y="Aspect Ratio", palette="mako", ax=axes[0])
axes[0].set_title("Video Aspect Ratio Distribution", fontweight="bold")
for i, count in enumerate(ar_counts["Count"]):
    axes[0].text(count + 20, i, f"{count:,} ({ar_counts['Percentage (%)'].iloc[i]}%)", va="center")

# Top exact resolutions
valid_meta["resolution_str"] = valid_meta["width"].astype(str) + "x" + valid_meta["height"].astype(str)
top_res = valid_meta["resolution_str"].value_counts().head(8).reset_index()
top_res.columns = ["Resolution", "Count"]
sns.barplot(data=top_res, x="Count", y="Resolution", palette="viridis", ax=axes[1])
axes[1].set_title("Top 8 Video Resolutions (WxH)", fontweight="bold")
for i, count in enumerate(top_res["Count"]):
    axes[1].text(count + 20, i, f"{count:,}", va="center")

plt.tight_layout()
fig_path = figures_dir / "video_resolution_distribution.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved figure: {fig_path}")


## 8. Frame Rate (FPS) and File Size Distribution

Examines video file sizes on disk (in MB) and framerate characteristics.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

# File size distribution
sns.histplot(valid_meta["file_size_mb"], bins=40, color="#2a9d8f", kde=True, ax=axes[0])
axes[0].set_title("Video File Size Distribution (MB)", fontweight="bold")
axes[0].set_xlabel("File Size (MB)")
axes[0].set_ylabel("Count")
axes[0].axvline(valid_meta["file_size_mb"].median(), color="red", linestyle="--",
                label=f"Median: {valid_meta['file_size_mb'].median():.2f} MB")
axes[0].legend()

# FPS distribution
top_fps = valid_meta["fps"].value_counts().head(8).reset_index()
top_fps.columns = ["FPS", "Count"]
top_fps["FPS"] = top_fps["FPS"].astype(str)
sns.barplot(data=top_fps, x="FPS", y="Count", palette="rocket", ax=axes[1])
axes[1].set_title("Framerate (FPS) Distribution", fontweight="bold")
axes[1].set_xlabel("Frames per Second (FPS)")
axes[1].set_ylabel("Count")

plt.tight_layout()
fig_path = figures_dir / "video_file_size_distribution.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved figure: {fig_path}")


## 9. Integrity Check and Outlier Reporting

We flag potential anomalies for dataset audit without modifying or deleting files on disk:
1. **Unreadable / Corrupt Videos**: Files where `is_valid == False` or `duration_sec <= 0`.
2. **Long Videos**: Videos exceeding standard short-form thresholds (> 180 seconds or > 300 seconds).

Flagged outliers are exported to `data/video_outliers_report.csv`.


In [ ]:
outliers_report_path = repo_root / "data" / "video_outliers_report.csv"

# Unreadable videos
unreadable = df_meta[~df_meta["is_valid"] | (df_meta["duration_sec"] <= 0)]
print(f"Number of unreadable/corrupted videos: {len(unreadable)}")
if len(unreadable) > 0:
    display(unreadable[["filename", "video_path", "error"]])

# Long videos (>180s and >300s)
long_180 = df_meta[df_meta["duration_sec"] > 180].copy()
long_300 = df_meta[df_meta["duration_sec"] > 300].copy()
print(f"Videos with duration > 180s (3 mins): {len(long_180)}")
print(f"Videos with duration > 300s (5 mins): {len(long_300)}")

# Compile outlier report
outlier_conditions = (
    (~df_meta["is_valid"]) |
    (df_meta["duration_sec"] <= 0) |
    (df_meta["duration_sec"] > 180)
)
df_outliers = df_meta[outlier_conditions].copy()
df_outliers["outlier_reason"] = np.where(
    ~df_outliers["is_valid"], "Unreadable / Corrupted",
    np.where(df_outliers["duration_sec"] > 300, "Duration > 300s", "Duration 180s - 300s")
)

df_outliers = df_outliers.sort_values(by="duration_sec", ascending=False)
df_outliers.to_csv(outliers_report_path, index=False)
print(f"Exported {len(df_outliers)} flagged outlier records to: {outliers_report_path}")

if len(df_outliers) > 0:
    print("Preview of top flagged outliers:")
    display(df_outliers[["filename", "duration_sec", "aspect_ratio", "file_size_mb", "outlier_reason"]].head(15))
